# Notebook 1: Identify Crop And Part With Router

This notebook runs only the router path: bootstrap, model access check, optional preloading and single-image analysis.

Notebook 1 is built for correct **plant/crop routing** on diseased images; it does not predict the disease class.

The router is prepared once per runtime and reused for later attempts, so models are not downloaded or reloaded for every new image.

The result keeps `crop` and `part` at the top level. The `router` block holds the router status, the primary detection and the detection count. This notebook does not load a crop adapter.

The default router profile is `balanced`, and diagnostic output (candidate crops, margin, unknown/rejection signals) is shown. This follows the open-set and selective-rejection literature: under uncertainty, a controlled decision is preferred over an overconfident label (Hendrycks & Gimpel, 2017; Guo et al., 2017; Geifman & El-Yaniv, 2019).


## Quick Flow

This notebook only finds crop/part with the router; it does not predict a disease class or run an adapter.

- For a first run, `NOTEBOOK_PROFILE = 'a100_colab_default'` and `IMAGE_PATH = None` are enough; the analysis cell asks for an upload.
- To try another image in the same runtime, re-run only the analysis cell.
- An `unknown` or `router_uncertain` result is not an error; it is the router's safe rejection.
- For adapter-based disease prediction, use Notebook 3 or Notebook 8.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/aads-open-world-plant-disease.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell01_bootstrap.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell02_access_check.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# User settings
NOTEBOOK_PROFILE = 'a100_colab_default'  # custom, a100_colab_default, cpu_debug
NOTEBOOK_SPEED_MODE = 'fast'  # fast, balanced, quality

# Optional last override layer
INFERENCE_OVERRIDES = {}

run_cell_script('nb1_cell03_runtime_setup.py', globals())

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# User settings
ANALYSIS_IMAGE_PATH = IMAGE_PATH  # Set a file path, or leave None to upload.

run_cell_script('nb1_cell04_analysis.py', globals())

result

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# User settings
ROUTER_EVAL_ROOT = None  # Example: '/content/router_part_eval'
ROUTER_EVAL_OUTPUT = None  # Example: '/content/router_part_eval_summary.json'
RUN_ROUTER_EVAL = False
EVAL_MIN_CONFIDENCE_GRID = '0.20,0.25,0.30,0.35,0.40,0.45,0.50'
EVAL_MARGIN_GRID = '0.00,0.02,0.05,0.08,0.10,0.12,0.15'

run_cell_script('nb1_cell05_router_eval.py', globals())